# ML Coding — Day 9: Quantization III (LLM methods)

**~1 hour, vectorized NumPy.** The techniques behind 4-bit LLMs: NF4, SmoothQuant, AWQ, GPTQ-style
error feedback, and QLoRA double-quant. Forward-only (these are PTQ methods). No element loops in
solutions (a loop over a small scale grid or weight columns is the algorithm and is allowed). Run the
**helpers cell first** — it defines the **NF4 codebook**.

**Q1** NF4 quant `[warmup]` · **Q2** SmoothQuant `[medium]` · **Q3** AWQ scale search `[medium]` ·
**Q4** GPTQ error feedback `[hard]` · **Q5** QLoRA double-quant `[hard · trick]`.

---

In [ ]:
# --- helpers (run me first) ---
import numpy as np

# QLoRA's NF4 codebook: 16 quantiles of a normal distribution (asymmetric, includes 0).
NF4 = np.array([-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224, 0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0])

## Q1 — NF4 quantization  ·  `[warmup]`

**Paper:** Dettmers et al., *QLoRA*, 2023 (arXiv:2305.14314). **Why it matters:** NF4 is a **non-uniform**
4-bit format whose 16 levels are the quantiles of a normal distribution — information-optimal for the
roughly-Gaussian weights of an LLM.

`nf4_quantize(x) -> (codes, scale)`: `scale = max|x|`; normalize to `[-1, 1]`; map each value to the
**nearest** of the 16 `NF4` levels (a `uint8` code). `nf4_dequantize(codes, scale) -> NF4[codes]·scale`.
No loops.

In [ ]:
import numpy as np

NF4 = np.array([-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224, 0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0])


def nf4_quantize(x):
    raise NotImplementedError


def nf4_dequantize(codes, scale):
    raise NotImplementedError

In [ ]:
# --- Q1 tests ---
def _q1():
    rng = np.random.default_rng(0)
    x = rng.standard_normal(64)
    codes, scale = nf4_quantize(x)
    assert codes.dtype == np.uint8 and codes.max() < 16
    xh = nf4_dequantize(codes, scale)
    assert np.max(np.abs(x - xh)) <= scale * 0.2         # within ~half the widest NF4 gap
    print("Q1 OK")

_q1()

## Q2 — SmoothQuant migration  ·  `[medium]`

**Paper:** Xiao et al., *SmoothQuant*, 2022 (arXiv:2211.10438). **Why it matters:** activations have
nasty per-channel outliers that wreck activation quantization; weights are easy. SmoothQuant **migrates**
the difficulty from activations into weights via a per-input-channel factor
`s_j = max|X_:,j|^α / max|W_j,:|^(1−α)`, dividing `X` and multiplying `W` — leaving `X·W` unchanged.

`smooth(X, W, alpha) -> (X_s, W_s, s)` with `X (N,Cin)`, `W (Cin,Cout)`. Verify `X_s @ W_s == X @ W`.
No loops.

In [ ]:
def smooth(X, W, alpha):
    raise NotImplementedError

In [ ]:
# --- Q2 tests ---
def _q2():
    rng = np.random.default_rng(1)
    X = rng.standard_normal((4, 8)) * 10                 # activation outliers
    W = rng.standard_normal((8, 5))
    Xs, Ws, s = smooth(X, W, 0.5)
    assert np.allclose(Xs @ Ws, X @ W)                   # mathematically equivalent
    assert np.max(np.abs(Xs)) <= np.max(np.abs(X))       # activation range shrunk
    print("Q2 OK")

_q2()

## Q3 — AWQ scale search  ·  `[medium]`

**Paper:** Lin et al., *AWQ*, 2023 (arXiv:2306.00978). **Why it matters:** not all weights matter
equally — the ones multiplying large activations are *salient*. AWQ scales weight channels by an
activation-derived factor before quantizing, protecting salient channels.

`awq_scale(X, W, ratios, qmax=127) -> (best_ratio, s)`: for each `ratio r`, set per-input-channel
`s = (max|X_:,j|)^r` (mean-normalized), quantize `W·s` per output channel, undo the scaling, and pick
the `r` minimizing output MSE vs `X@W`. (Looping the small `ratios` grid is fine.)

In [ ]:
def awq_scale(X, W, ratios, qmax=127):
    raise NotImplementedError

In [ ]:
# --- Q3 tests ---
def _q3():
    rng = np.random.default_rng(2)
    X = rng.standard_normal((8, 6)); X[:, 0] *= 20       # channel 0 is salient
    W = rng.standard_normal((6, 4))
    r, s = awq_scale(X, W, np.linspace(0, 1, 11))
    Yf = X @ W

    def out_mse(scal):
        Ws = W * scal[:, None]
        sw = np.where(np.max(np.abs(Ws), 0) / 127 > 0, np.max(np.abs(Ws), 0) / 127, 1.0)
        Wq = np.clip(np.round(Ws / sw), -127, 127) * sw
        return np.mean((Yf - X @ (Wq / scal[:, None])) ** 2)

    assert out_mse(s) <= out_mse(np.ones(6)) + 1e-9      # no worse than no scaling (ratio 0)
    print("Q3 OK")

_q3()

## Q4 — GPTQ-style error feedback  ·  `[hard]`

**Paper:** Frantar et al., *GPTQ*, 2022 (arXiv:2210.17323). **Why it matters:** quantizing weights
independently lets errors accumulate. GPTQ quantizes column-by-column and **pushes each column's
residual into the columns not yet quantized**, so errors cancel rather than pile up (a simplified,
Hessian-free version here).

`gptq_quantize(W, scale, qmax=127) -> W_q`: for each column (left→right), add the carried error, round
to the grid, and carry the new residual forward. (Column loop is the algorithm.)

In [ ]:
def gptq_quantize(W, scale, qmax=127):
    raise NotImplementedError

In [ ]:
# --- Q4 tests ---
def _q4():
    rng = np.random.default_rng(3)
    W = rng.standard_normal((6, 8)) * 0.1
    scale = 0.05
    Wq = gptq_quantize(W, scale)
    Wn = np.clip(np.round(W / scale), -127, 127) * scale         # naive round-to-nearest
    x = np.ones(8)
    err_gptq = np.sum(np.abs(W @ x - Wq @ x))
    err_naive = np.sum(np.abs(W @ x - Wn @ x))
    assert err_gptq <= err_naive + 1e-9                          # error feedback cancels, not piles up
    print("Q4 OK")

_q4()

## Q5 — QLoRA double-quantization  ·  `[hard · tensor trick]`

**Paper:** Dettmers et al., *QLoRA*, 2023. **The trick:** block-wise NF4 needs one float `scale` per
block — at block size 64 that's a non-trivial overhead, so QLoRA **quantizes the scales too** (a second
level). Implement `double_quant(x, block) -> (codes, q_scales, meta_scale)`: split into blocks, NF4-code
each block with its `max|·|` scale, then quantize the vector of block scales to `uint8` with a single
`meta_scale`. `double_dequant(codes, q_scales, meta_scale, block) -> x_hat` reverses both levels.

**Plan:** reshape to `(nblocks, block)`, NF4 over the last axis, then a plain affine quant of the
`(nblocks,)` scale vector. No loops.

In [ ]:
def double_quant(x, block):
    raise NotImplementedError


def double_dequant(codes, q_scales, meta_scale, block):
    raise NotImplementedError

In [ ]:
# --- Q5 tests ---
def _q5():
    rng = np.random.default_rng(4)
    x = rng.standard_normal(128)
    codes, q_scales, meta = double_quant(x, block=32)
    assert codes.shape == (4, 32) and q_scales.shape == (4,) and codes.dtype == np.uint8
    xh = double_dequant(codes, q_scales, meta, 32)
    assert xh.shape == (128,) and np.max(np.abs(x - xh)) < 0.6   # NF4 + scale-quant error, bounded
    print("Q5 OK")

_q5()

## Likely follow-ups
- Why NF4 beats uniform int4 on Gaussian weights; GGUF k-quants (super-block scales).
- SmoothQuant α tuning; combining SmoothQuant + per-channel weight quant.
- GPTQ's real Hessian-based update (OBQ / Cholesky); activation-order (act-order) quantization.

---
## Reference solutions (don't peek until you've tried)

In [ ]:
import numpy as np

NF4 = np.array([-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224, 0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0])


def ref_nf4_quantize(x):
    x = np.asarray(x, float)
    scale = np.max(np.abs(x))
    scale = scale if scale > 0 else 1.0
    xn = x / scale
    codes = np.argmin(np.abs(xn[..., None] - NF4[None, :]), axis=-1).astype(np.uint8)
    return codes, scale


def ref_nf4_dequantize(codes, scale):
    return NF4[codes] * scale


def ref_smooth(X, W, alpha):
    X = np.asarray(X, float); W = np.asarray(W, float)
    act_max = np.max(np.abs(X), axis=0)                  # per input channel
    w_max = np.max(np.abs(W), axis=1)
    s = (act_max ** alpha) / (w_max ** (1 - alpha))
    s = np.where(s > 0, s, 1.0)
    return X / s[None, :], W * s[:, None], s


def ref_awq_scale(X, W, ratios, qmax=127):
    X = np.asarray(X, float); W = np.asarray(W, float)
    act = np.max(np.abs(X), axis=0)
    act = np.where(act > 0, act, 1.0)
    Yf = X @ W
    best_r, best_s, best_mse = None, None, np.inf
    for r in ratios:
        s = act ** r
        s = s / np.mean(s)                               # normalize (no free global scaling)
        Ws = W * s[:, None]
        sw = np.max(np.abs(Ws), axis=0) / qmax
        sw = np.where(sw > 0, sw, 1.0)
        Wq = np.clip(np.round(Ws / sw), -qmax, qmax) * sw
        mse = np.mean((Yf - X @ (Wq / s[:, None])) ** 2)
        if mse < best_mse:
            best_mse, best_r, best_s = mse, r, s
    return best_r, best_s


def ref_gptq_quantize(W, scale, qmax=127):
    W = np.asarray(W, float)
    Wq = np.zeros_like(W)
    err = np.zeros(W.shape[0])
    for c in range(W.shape[1]):
        col = W[:, c] + err                              # carry residual from earlier columns
        q = np.clip(np.round(col / scale), -qmax, qmax) * scale
        Wq[:, c] = q
        err = col - q
    return Wq


def ref_double_quant(x, block):
    x = np.asarray(x, float)
    xb = x.reshape(-1, block)
    scales = np.max(np.abs(xb), axis=1)
    scales = np.where(scales > 0, scales, 1.0)
    xn = xb / scales[:, None]
    codes = np.argmin(np.abs(xn[..., None] - NF4[None, None, :]), axis=-1).astype(np.uint8)
    meta = np.max(scales) / 255.0
    meta = meta if meta > 0 else 1.0
    q_scales = np.clip(np.round(scales / meta), 0, 255).astype(np.uint8)
    return codes, q_scales, meta


def ref_double_dequant(codes, q_scales, meta_scale, block):
    scales = q_scales.astype(float) * meta_scale
    return (NF4[codes] * scales[:, None]).reshape(-1)


_x = np.random.default_rng(7).standard_normal(64)
_c, _s = ref_nf4_quantize(_x); assert _c.dtype == np.uint8 and np.max(np.abs(_x - ref_nf4_dequantize(_c, _s))) <= _s * 0.2
_X = np.random.default_rng(8).standard_normal((4, 6)); _W = np.random.default_rng(9).standard_normal((6, 3))
_Xs, _Ws, _ss = ref_smooth(_X, _W, 0.5); assert np.allclose(_Xs @ _Ws, _X @ _W)
_Wq = ref_gptq_quantize(_W, 0.05); assert _Wq.shape == _W.shape
_co, _qs, _m = ref_double_quant(np.random.default_rng(1).standard_normal(96), 32)
assert _co.shape == (3, 32) and _qs.shape == (3,)
print("reference solutions OK")